## Week 8 Tutorial Part-2: Object Detection Using SSD

## Welcome to the 8th Lab of 42028: Deep Learning and CNN!

In this Lab/Tutorial session, you will learn how to train an object detection model for a custom dataset and evaluate its performance on a test dataset.

In this lab, you will focus on the configuration of the SSD object detector for a custom dataset.

Let's get started!

## Tutorial:

1. Image annotation using CVAT: Computer Vision Annotation Tool
   (Reference: https://cvat.ai)
2. Creation of the Data Loaders for the new dataset
3. Training the model
4. Evaluating the model
5. Using the trained model for inference

## Tasks for this week:

1. Install dependancies
2. Install libraries and download the dataset
3. Train/Finetune the model using transfer learning from pre-trained models and evaluate it
4. Use a trained model for inference

## Install Requirements



In [ ]:
!pip install torch torchvision -q
!pip install pycocotools lxml tqdm -q

In [ ]:
!python --version

## Make sure the dataset has this structure, PASCAL VOC XML

```
data/images/train/
    0.jpg
    0.xml
    1.jpg
    1.xml
    ...
data/images/valid/
    0.jpg
    0.xml
    1.jpg
    1.xml
    ...
data/images/test/
    0.jpg
    0.xml
    1.jpg
    1.xml
    ...
```

## Unzip the dataset

In [ ]:
# Uncomment the below code if the data.zip was not unzipped in the previsous task
# !pwd
# !unzip data.zip

## Define Data Directory Paths

In [ ]:
# WRITE YOUR CODE
TRAIN_DIR = ''  # Update it to the correct path to the train/test/valid directory
VAL_DIR = ''
TEST_DIR = '' # Use validation dataset because we do not have a test set - you may have one, if it has annotations, you may evaluate


## Imports

In [ ]:
# STEP 3: Imports
import os
import xml.etree.ElementTree as ET
import torch
from torchvision.transforms import functional as F
from PIL import Image
import torchvision
from torchvision.ops import box_iou
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
import numpy as np
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

## Check if GPU is available.

IF NO GPU - YOU WILL HAVE TO WAIT FOR LONGER AMOUNT OF TIME TO TRAIN, SO MAKE SURE YOU'RE ON A GPU RUNTIME.

In [ ]:
torch.cuda.is_available()

## Create the Class list

In [ ]:
# Class list (must match your dataset/annotations for proper index and number of classes)
# faster RCNN needs a background class named `__background__`
# WRITE YOUR CODE
CLASSES = ['__background__', '', '', '']
NUM_CLASSES = len(CLASSES)

## Parse VOC XML annotations

In [ ]:
def parse_voc_xml(xml_file):
    # Parse the XML annotation file using ElementTree
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Initialize lists to store bounding boxes and labels
    boxes, labels = [], []

    # Loop over all object elements in the XML
    for obj in root.findall("object"):
        # Get the object class name
        label = obj.find("name").text

        # Skip labels that are not in the defined CLASSES list
        if label not in CLASSES:
            continue

        # Convert label name to its corresponding index in CLASSES
        labels.append(CLASSES.index(label))

        # Extract the bounding box coordinates from the XML
        bbox = obj.find("bndbox")
        box = [
            float(bbox.find("xmin").text),  # left
            float(bbox.find("ymin").text),  # top
            float(bbox.find("xmax").text),  # right
            float(bbox.find("ymax").text)   # bottom
        ]
        boxes.append(box)

    # Return list of bounding boxes and their corresponding labels
    return boxes, labels


## Make a custom VOC Dataset class to load dataset

In [ ]:
class VOCDataset(Dataset):
    def __init__(self, root_dir, transforms=None):
        # Set root directory where images and annotations are stored
        self.root_dir = root_dir
        self.transforms = transforms

        # Get list of all image filenames ending with .jpg
        self.images = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]

        # Sort filenames to maintain consistent ordering
        self.images.sort()

    def __getitem__(self, idx):
        # Get the image filename
        img_name = self.images[idx]

        # Build full paths for the image and its corresponding annotation file
        img_path = os.path.join(self.root_dir, img_name)
        xml_path = img_path.replace('.jpg', '.xml')

        # Load image and convert to RGB
        img = Image.open(img_path).convert("RGB")

        # Parse XML annotation to get bounding boxes and labels
        boxes, labels = parse_voc_xml(xml_path)

        # Convert to torch tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        # Build target dictionary
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        # Apply transforms if any
        if self.transforms:
            img = self.transforms(img)
        else:
            img = F.to_tensor(img)

        return img, target

    def __len__(self):
        # Return total number of images
        return len(self.images)


# Download a pre-trained model and modify to make it trainable

Note: In FasterRCNN we used pre-trained weights for backbone as well as the head, here we are using pre-trained weights for backbone only, because SSD needs to dynamically create the head based on number of classes.

In [ ]:
def get_ssd_model(num_classes):
    # Load an SSD model with:
    # - NO pretrained head (weights=None) → because the default head is for COCO (91 classes)
    # - YES pretrained backbone (weights_backbone="DEFAULT") → reuse MobileNetV3-Large features
    # - Custom number of classes → builds a new classification head for your dataset

    model = ssdlite320_mobilenet_v3_large(
        weights=None,                 # Avoid loading full COCO-trained model (wrong head)
        weights_backbone="DEFAULT",  # Load pretrained MobileNetV3-Large backbone
        num_classes=num_classes      # Replace the classification head with a new one for your task
    )

    return model


## DataLoader setup

In [ ]:
# Create the training dataset
dataset = VOCDataset(TRAIN_DIR)

# Create a DataLoader with custom collate function for handling variable-size targets
data_loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=lambda x: tuple(zip(*x))
)

# Set device to GPU if available, otherwise fallback to CPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Load the SSD model and move it to the selected device
model = get_ssd_model(NUM_CLASSES).to(device)

## Utility to save a trained model for future - will use it for checkpointing

In [ ]:
import torch

def save_model(model, optimizer, epoch, filename="checkpoint.pth"):
    """
    Saves the model and optimizer state for later training or inference.
    Args:
        model (torch.nn.Module): The model to save.
        optimizer (torch.optim.Optimizer): The optimizer used during training.
        epoch (int): Current training epoch.
        filename (str): File name to save the checkpoint.
    """
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, filename)
    print(f"✅ Saved model checkpoint to: {filename}")


## Define Metric computation utility function (per COCO style)

We will use this to evaluate as well as validate while training

In [ ]:
def compute_map_ar(preds, targets, num_classes=len(CLASSES)-1):
    # Initialize the results dictionary with default values
    results = {
        'map': 0, 'map_50': 0, 'map_75': 0,
        'map_per_class': torch.zeros(num_classes),
        'mar_1': 0, 'mar_10': 0, 'mar_100': 0,
        'mar_100_per_class': torch.zeros(num_classes),
    }

    # Lists to hold AP and AR values for each class
    aps = [[] for _ in range(num_classes)]
    ars = [[] for _ in range(num_classes)]

    # Loop through each image's predictions and targets
    for pred, target in zip(preds, targets):
        # Loop through each class (excluding background)
        for class_idx in range(1, num_classes+1):
            # Filter boxes by current class
            gt_mask = target['labels'] == class_idx
            pred_mask = pred['labels'] == class_idx

            gt_boxes = target['boxes'][gt_mask]
            pred_boxes = pred['boxes'][pred_mask]
            pred_scores = pred['scores'][pred_mask]

            # Skip if no GT or predictions
            if len(gt_boxes) == 0 and len(pred_boxes) == 0:
                continue

            # Compute IoUs between predictions and ground truth
            ious = box_iou(pred_boxes, gt_boxes) if len(gt_boxes) > 0 and len(pred_boxes) > 0 else torch.zeros((0, 0))

            # Initialize true positives (TP) and matched GT indices
            tp = torch.zeros(len(pred_boxes))
            matched = set()

            # Match predictions to ground truth based on IoU > 0.5
            for i, row in enumerate(ious):
                max_iou, max_j = torch.max(row, dim=0)
                if max_iou > 0.5 and max_j.item() not in matched:
                    tp[i] = 1
                    matched.add(max_j.item())

            # Compute false positives (FP)
            fp = 1 - tp

            # Cumulative TP and FP for precision-recall curve
            cum_tp = torch.cumsum(tp, dim=0)
            cum_fp = torch.cumsum(fp, dim=0)

            # Compute recall and precision
            recalls = cum_tp / (len(gt_boxes) + 1e-6)
            precisions = cum_tp / (cum_tp + cum_fp + 1e-6)

            # Compute AP (area under precision-recall curve)
            ap = torch.trapz(precisions, recalls) if recalls.numel() > 0 else torch.tensor(0.)
            # AR is the max recall value
            ar = recalls[-1] if recalls.numel() > 0 else torch.tensor(0.)

            # Store per-class AP and AR
            aps[class_idx-1].append(ap.item())
            ars[class_idx-1].append(ar.item())

    # Compute average AP and AR for each class
    ap_avg = torch.tensor([np.mean(cls_ap) if cls_ap else 0. for cls_ap in aps])
    ar_avg = torch.tensor([np.mean(cls_ar) if cls_ar else 0. for cls_ar in ars])

    # Save results
    results['map_per_class'] = ap_avg
    results['mar_100_per_class'] = ar_avg
    results['map'] = ap_avg.mean()
    results['map_50'] = ap_avg.mean()
    results['map_75'] = ap_avg.mean()
    results['mar_100'] = ar_avg.mean()
    results['mar_10'] = ar_avg.mean()
    results['mar_1'] = ar_avg.mean()

    return results


## Define Evaluation utilities (COCO-style)

We will use this to evaluate as well as validate while training

In [ ]:
def evaluate_map(model, dataset, iou_thresholds=[0.5, 0.75]):
    # Set model to evaluation mode
    model.eval()

    # Lists to store predictions and ground truth for all images
    all_preds, all_targets = [], []

    # Loop over all images in the dataset
    for img, target in dataset:
        # Add batch dimension and move image to device
        img = img.to(device).unsqueeze(0)

        # Run inference without gradients
        with torch.no_grad():
            pred = model(img)[0]

        # Filter out predictions with low confidence scores
        keep = pred['scores'] > 0.05
        pred_boxes = pred['boxes'][keep].cpu()
        pred_labels = pred['labels'][keep].cpu()
        pred_scores = pred['scores'][keep].cpu()

        # Store filtered predictions
        all_preds.append({
            'boxes': pred_boxes,
            'labels': pred_labels,
            'scores': pred_scores
        })

        # Store ground truth (converted to CPU)
        all_targets.append({
            'boxes': target['boxes'].cpu(),
            'labels': target['labels'].cpu()
        })

    # Compute mAP and AR metrics using predictions and targets
    return compute_map_ar(all_preds, all_targets)


## Training

### Setup – Optimizer, Epochs, Logging

In [ ]:
# Define the optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

# Number of epochs to train
num_epochs = 50  # SSD needs more epochs than FasterRCNN

start_epoch = 0 # this is changed if you load a pre-trained model or checkpoint to continue training

# Logging and checkpointing settings
# Depending on the batch size and number of images in training set as num_iterations = num_images/batch_size
# Our batch size is 8, num_images in train = 16, so num_iter = 2, and we can see this in the result output with print_every = 1.
print_every = 1        # Print loss every N batches
save_every = 1         # Save model every N epochs
val_every = 1          # Validate every N epochs
save_dir = './SSDtraining' # WRITE/UPDATE THE PATH TO TRAINING DIRECTORY
os.makedirs(save_dir, exist_ok=True)

# Initialize logging lists
epoch_losses = []
iteration_losses = []
val_maps = []

val_dataset = VOCDataset(VAL_DIR) # setup val data loader for validation



### Training Loop

In [ ]:
for epoch in range(start_epoch, num_epochs):
    model.train()  # Set model to training mode
    total_loss = 0  # Track total loss for the epoch

    for i, (images, targets) in enumerate(data_loader):
        # Move all images and targets to the selected device
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Get the loss dict from the model
        loss_dict = model(images, targets)

        # Combine all losses into a single scalar
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass and optimizer step
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        # Accumulate batch loss and append to losses per iteration
        total_loss += losses.item()
        iteration_losses.append(losses.item())
        # Print batch loss every few iterations
        if (i + 1) % print_every == 0:
            print(f"  [Epoch {epoch+1}, Iter {i+1}] Loss: {losses.item():.4f}")

    # Print total loss at the end of the epoch and store losses for loss curve
    print(f"Epoch [{epoch+1}/{num_epochs}], Total Loss: {total_loss:.4f}")
    epoch_losses.append(total_loss)

    if (epoch+1) % val_every == 0:
      # Evaluate on validation set
      val_results = evaluate_map(model, val_dataset)
      val_map = val_results["map"].item() # feel free to add more for per map/mar graph (@50, @75, etc)
      val_maps.append(val_map)
      print(f"📈 Validation mAP at epoch {epoch+1}: {val_map:.4f}")

    # Save checkpoint for loss every few epochs
    if (epoch + 1) % save_every == 0:
      save_model(model, optimizer, epoch+1, filename=str(os.path.join(save_dir, f"checkpoint_ssd_epoch_{epoch+1}.pth")))

# save final model after training is complete
save_model(model, optimizer, epoch+1, filename=str(os.path.join(save_dir,f"final_model_ssd_epoch_{epoch+1}.pth")))

### Plot loss and map50 curves

In [ ]:
import matplotlib.pyplot as plt

# Plot training loss per epoch
plt.figure(figsize=(10, 4))
plt.plot(epoch_losses, marker='o')
plt.title("Training Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# Plot validation mAP per epoch
plt.figure(figsize=(10, 4))
plt.plot(val_maps, marker='s', color='green')
plt.title("Validation mAP per Epoch")
plt.xlabel("Epoch")
plt.ylabel("mAP (IoU=1.0)")
plt.grid(True)
plt.show()


## Print model summary

In [ ]:
print(model)

## Inference on a sample image

In [ ]:
# Set the model to evaluation mode
model.eval()

# Pick an image path from the training set
img_path = os.path.join(TRAIN_DIR, dataset.images[0])

# Open the image and convert to RGB
img = Image.open(img_path).convert("RGB")

# Convert the image to a tensor and add a batch dimension
img_tensor = F.to_tensor(img).unsqueeze(0).to(device)

# Run the model in inference mode without computing gradients
with torch.no_grad():
    output = model(img_tensor)[0]  # Get predictions for the first image


## Visualize predictions

Note: EXPERIMENT WITH VARIOUS THRESHOLDS

In [ ]:
# Create a figure to display the image
plt.figure(figsize=(10, 6))
plt.imshow(img)  # Show the original image
ax = plt.gca()   # Get the current axes

# Loop through predicted boxes, labels, and scores
for box, label, score in zip(output['boxes'], output['labels'], output['scores']):
    # Note experiment with 0.3, 0.7, 0.8, 0.9
    if score > 0.5:  # Only show predictions above a confidence threshold
        # Extract box coordinates
        x1, y1, x2, y2 = box.cpu().numpy()

        # Draw the bounding box
        ax.add_patch(plt.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            edgecolor='red', facecolor='none', linewidth=2
        ))

        # Draw the label and score
        ax.text(
            x1, y1,
            f"{CLASSES[label]}: {score:.2f}",
            color='white',
            bbox=dict(facecolor='red', alpha=0.5)
        )

# Hide axis ticks
plt.axis('off')
plt.show()


## Run Evaluation on TRAIN set

In [ ]:
# Evaluate the model on the training dataset
train_results = evaluate_map(model, dataset)

# Print overall mAP/mAR results for the training set
print("📊 Train set mAP/mAR results:")
for k, v in train_results.items():
    # Convert torch tensors to NumPy arrays for clean printing
    if isinstance(v, torch.Tensor):
        print(f"{k}: {v.numpy()}")
    else:
        print(f"{k}: {v}")


### Pretty-print AP/AR per class

In [ ]:
# Print header for per-class AP / AR table
print("\nAP / AR per class\n" + "-"*73)
print(f"| {'ID':<3} | {'Class':<20} | {'AP':<18} | {'AR':<18} |")
print("-"*73)

# Loop through each class (excluding background)
for i, cls in enumerate(CLASSES[1:], 1):
    # Get AP and AR for the current class
    ap = train_results['map_per_class'][i-1].item()
    ar = train_results['mar_100_per_class'][i-1].item()

    # Print formatted row
    print(f"| {i:<3} | {cls:<20} | {ap:<18.3f} | {ar:<18.3f} |")

# Print average AP and AR across all classes
print("-"*73)
print(f"| {'Avg':<24} | {train_results['map'].item():<18.3f} | {train_results['mar_100'].item():<18.3f} |")
print("-"*73)


## Run Evaluation on VALIDATION set

### Create Validation set dataloader

In [ ]:
# Create the validation dataset
val_dataset = VOCDataset(VAL_DIR)

# Create a DataLoader for the validation set
val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,  # No shuffling for evaluation
    collate_fn=lambda x: tuple(zip(*x))  # Custom collate function to handle varying targets
)


### Evaluate on validation set


In [ ]:
# Evaluate the model on the validation dataset
val_results = evaluate_map(model, val_dataset)

# Print overall mAP/mAR results for the validation set
print("\n📊 Validation set mAP/mAR results:")
for k, v in val_results.items():
    # Convert tensors to NumPy arrays for cleaner output
    if isinstance(v, torch.Tensor):
        print(f"{k}: {v.numpy()}")
    else:
        print(f"{k}: {v}")


### Pretty-print AP/AR per class

In [ ]:
# Print header for per-class AP / AR table on validation set
print("\nAP / AR per class on validation\n" + "-"*73)
print(f"| {'ID':<3} | {'Class':<20} | {'AP':<18} | {'AR':<18} |")
print("-"*73)

# Loop through each class (excluding background)
for i, cls in enumerate(CLASSES[1:], 1):
    # Get per-class AP and AR from results
    ap = val_results['map_per_class'][i-1].item()
    ar = val_results['mar_100_per_class'][i-1].item()

    # Print table row for the class
    print(f"| {i:<3} | {cls:<20} | {ap:<18.3f} | {ar:<18.3f} |")

# Print average AP and AR across all classes
print("-"*73)
print(f"| {'Avg':<24} | {val_results['map'].item():<18.3f} | {val_results['mar_100'].item():<18.3f} |")
print("-"*73)


## Run Evaluation on TEST set (if available with annotations)

### Create Test set dataloader

In [ ]:
# Create the test dataset
test_dataset = VOCDataset(TEST_DIR)

# Create a DataLoader for the test set
test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,  # No shuffling needed for evaluation
    collate_fn=lambda x: tuple(zip(*x))  # Handle varying-size targets
)


### Evaluate on test set


In [ ]:
# Evaluate the model on the test dataset
test_results = evaluate_map(model, test_dataset)

# Print overall mAP/mAR results for the test set
print("\n📊 Test set mAP/mAR results:")
for k, v in test_results.items():
    # Convert torch tensors to NumPy arrays for cleaner display
    if isinstance(v, torch.Tensor):
        print(f"{k}: {v.numpy()}")
    else:
        print(f"{k}: {v}")


### Pretty-print AP/AR per class

In [ ]:
# Print table header for per-class AP and AR
print("\nAP / AR per class on test\n" + "-"*73)
print(f"| {'ID':<3} | {'Class':<20} | {'AP':<18} | {'AR':<18} |")
print("-"*73)

# Loop through each class (excluding background)
for i, cls in enumerate(CLASSES[1:], 1):
    # Get AP and AR for the current class
    ap = test_results['map_per_class'][i-1].item()
    ar = test_results['mar_100_per_class'][i-1].item()

    # Print table row for the class
    print(f"| {i:<3} | {cls:<20} | {ap:<18.3f} | {ar:<18.3f} |")

# Print average AP and AR across all classes
print("-"*73)
print(f"| {'Avg':<24} | {test_results['map'].item():<18.3f} | {test_results['mar_100'].item():<18.3f} |")
print("-"*73)


## Utility to load a trained model for future use

In [ ]:
def load_model(model, optimizer=None, filename="checkpoint.pth", device="cpu"):
    """
    Loads a model (and optionally optimizer) from a saved checkpoint.
    Args:
        model (torch.nn.Module): The model architecture to load weights into.
        optimizer (optional): The optimizer to load state into (if continuing training).
        filename (str): Path to the saved checkpoint.
        device (str or torch.device): Device to map the model to.
    Returns:
        model, optimizer (if provided), start_epoch
    """
    checkpoint = torch.load(filename, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)

    start_epoch = checkpoint.get('epoch', 0)

    if optimizer:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        return model, optimizer, start_epoch

    return model, start_epoch


In [ ]:
# Save after training
save_model(model, optimizer, epoch=num_epochs, filename="./SSDtraining/model_ssd.pth") # WRITE/UPDATE THE PATH TO TRAINING DIRECTORY

In [ ]:
# Rebuild model architecture
model = get_ssd_model(NUM_CLASSES)

# Load the model (without optimizer if you're not training)
model, start_epoch = load_model(model, filename="./SSDtraining/model_ssd.pth", device=device) # WRITE/UPDATE THE PATH TO TRAINING DIRECTORY

# Set to eval mode for inference
model.eval()


### Feel free to use this start_epoch to resume training... and continue training below....